# End-To-End Memory Networks (MemN2N) - bAbI QA

This notebook implements the **End-To-End Memory Network (MemN2N)** from *Sukhbaatar et al., NIPS 2015* and trains it on the **full bAbI QA benchmark (tasks 1–20)**.

## Notebook summary
- Download + parse bAbI v1.1
- Vectorize stories/questions/answers
- Train MemN2N with multi-hop memory attention
- Use position encoding and temporal encoding
- Evaluate overall test accuracy and per-task accuracy
- Inspect attention weights on one example

## Project group members:
- Juan Kenichi SUTAN
- Maxime HAYAKAWA-IVANOVIC

In [1]:
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from babi_utils import (
    build_vocab,
    download_and_extract_babi,
    infer_max_sizes,
    load_babi_tasks,
    vectorize_examples_with_task_ids,
)

print("torch:", torch.__version__)
print("device:", "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))


torch: 2.8.0
device: mps


In [2]:
# Repro
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
)

# bAbI scope: full benchmark (tasks 1..20)
TASK_IDS = list(range(1, 21))
BABI_SUBSET = "en-10k"  # train/test split; dev is created from train when missing

# Tensorization defaults (paper uses 50 most recent sentences)
MEMORY_SIZE_CAP = 50
SENTENCE_SIZE_CAP = 20
QUESTION_SIZE_CAP = 20

# Training defaults
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-3
CLIP_NORM = 40.0

# Model defaults
D_EMB = 50
K_HOPS = 3

USE_POSITION_ENCODING = True
USE_TEMPORAL_ENCODING = True

print("TASK_IDS:", TASK_IDS)


TASK_IDS: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## Data: bAbI QA (download → parse → vectorize)

We use the bAbI v1.1 question answering dataset with weak supervision (supporting-fact labels are not used).

This notebook uses all 20 tasks together.


In [3]:
DATA_DIR = Path(".") / "data"
BABI_ROOT = download_and_extract_babi(DATA_DIR, extracted_dirname="babi")

train_ex, valid_ex, test_ex = load_babi_tasks(BABI_ROOT, TASK_IDS, subset=BABI_SUBSET)

# Some distributions (e.g. en-10k) do not ship a *_valid.txt split.
# In that case, we create a small dev split from train (paper holds out ~10%).
if len(valid_ex) == 0:
    rng = np.random.default_rng(SEED)
    idx = np.arange(len(train_ex))
    rng.shuffle(idx)
    dev_n = max(1, int(0.1 * len(train_ex)))
    dev_idx = set(idx[:dev_n].tolist())
    valid_ex = [ex for i, ex in enumerate(train_ex) if i in dev_idx]
    train_ex = [ex for i, ex in enumerate(train_ex) if i not in dev_idx]

print("#train:", len(train_ex), "#valid:", len(valid_ex), "#test:", len(test_ex))

# Build vocab on all splits for the selected tasks (simple and robust)
stoi, itos = build_vocab(list(train_ex) + list(valid_ex) + list(test_ex))
V = len(stoi)
print("Vocab size:", V)

# Infer max sizes with caps
mem_size, sent_size, q_size = infer_max_sizes(
    list(train_ex) + list(valid_ex) + list(test_ex),
    memory_cap=MEMORY_SIZE_CAP,
    sentence_cap=SENTENCE_SIZE_CAP,
    question_cap=QUESTION_SIZE_CAP,
)
print("memory_size:", mem_size, "sentence_size:", sent_size, "question_size:", q_size)

Xtr_m, Xtr_q, ytr, ttr = vectorize_examples_with_task_ids(
    train_ex, stoi, memory_size=mem_size, sentence_size=sent_size, question_size=q_size
)
Xva_m, Xva_q, yva, tva = vectorize_examples_with_task_ids(
    valid_ex, stoi, memory_size=mem_size, sentence_size=sent_size, question_size=q_size
)
Xte_m, Xte_q, yte, tte = vectorize_examples_with_task_ids(
    test_ex, stoi, memory_size=mem_size, sentence_size=sent_size, question_size=q_size
)

# Torch datasets (include task_id for per-task evaluation)
train_ds = TensorDataset(torch.from_numpy(Xtr_m), torch.from_numpy(Xtr_q), torch.from_numpy(ytr), torch.from_numpy(ttr))
valid_ds = TensorDataset(torch.from_numpy(Xva_m), torch.from_numpy(Xva_q), torch.from_numpy(yva), torch.from_numpy(tva))
test_ds = TensorDataset(torch.from_numpy(Xte_m), torch.from_numpy(Xte_q), torch.from_numpy(yte), torch.from_numpy(tte))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

PAD_ID = stoi["<PAD>"]
UNK_ID = stoi["<UNK>"]
print("PAD_ID:", PAD_ID, "UNK_ID:", UNK_ID)


#train: 180000 #valid: 20000 #test: 20000
Vocab size: 183
memory_size: 50 sentence_size: 11 question_size: 11
PAD_ID: 0 UNK_ID: 1


## Sentence encoders: BoW and Position Encoding (PE)

- BoW: sum token embeddings.
- PE (paper Section 4.1): sum token embeddings weighted by a fixed position-dependent matrix.

Both encoders take token ids shaped `[batch, ..., J]` and produce vectors shaped `[batch, ..., d]`.


In [4]:
def position_encoding_matrix(sentence_size: int, d: int, device: torch.device) -> torch.Tensor:
    """Return PE matrix l of shape [sentence_size, d] (paper Section 4.1)."""
    J = sentence_size
    D = d

    j = torch.arange(1, J + 1, device=device, dtype=torch.float32).view(J, 1)  # [J,1]
    k = torch.arange(1, D + 1, device=device, dtype=torch.float32).view(1, D)  # [1,D]

    l = (1.0 - j / J) - (k / D) * (1.0 - 2.0 * j / J)
    return l  # [J,D]


def encode_bow(emb: nn.Embedding, tokens: torch.Tensor) -> torch.Tensor:
    """BoW encoding: sum embeddings over last dimension (token axis)."""
    x = emb(tokens)  # [..., J, d]
    return x.sum(dim=-2)


def encode_pe(emb: nn.Embedding, tokens: torch.Tensor, pe: torch.Tensor) -> torch.Tensor:
    """PE encoding: sum (embedding * l_j) over tokens."""
    # tokens: [..., J]
    x = emb(tokens)  # [..., J, d]

    # pe: [J, d] -> broadcast to x
    while pe.dim() < x.dim():
        pe = pe.unsqueeze(0)
    x = x * pe
    return x.sum(dim=-2)


# Quick shape sanity check
with torch.no_grad():
    m, q, y, tid = next(iter(train_loader))
    m = m.to(DEVICE)
    q = q.to(DEVICE)

    test_emb = nn.Embedding(V, 8, padding_idx=PAD_ID).to(DEVICE)
    pe_test = position_encoding_matrix(sent_size, 8, DEVICE)

    m_bow = encode_bow(test_emb, m)  # [B, I, d]
    q_bow = encode_bow(test_emb, q)  # [B, d]
    m_pe = encode_pe(test_emb, m, pe_test)
    q_pe = encode_pe(test_emb, q, position_encoding_matrix(q_size, 8, DEVICE))

    print("m tokens:", m.shape, "-> bow:", m_bow.shape, "pe:", m_pe.shape)
    print("q tokens:", q.shape, "-> bow:", q_bow.shape, "pe:", q_pe.shape)


m tokens: torch.Size([32, 50, 11]) -> bow: torch.Size([32, 50, 8]) pe: torch.Size([32, 50, 8])
q tokens: torch.Size([32, 11]) -> bow: torch.Size([32, 8]) pe: torch.Size([32, 8])


## MemN2N model (multi-hop attention over external memory)

We implement the core equations from the paper:

- $p_i = \mathrm{softmax}(u^\top m_i)$
- $o = \sum_i p_i c_i$
- $u \leftarrow u + o$ for each hop

We use adjacent weight tying and temporal encoding.


In [5]:
class MemN2N(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d: int,
        memory_size: int,
        sentence_size: int,
        question_size: int,
        hops: int,
        *,
        pad_id: int,
        use_position_encoding: bool = True,
        use_temporal_encoding: bool = True,
        tie_out: bool = True,
    ) -> None:
        super().__init__()
        self.V = vocab_size
        self.d = d
        self.I = memory_size
        self.J = sentence_size
        self.Jq = question_size
        self.K = hops

        self.pad_id = pad_id
        self.use_position_encoding = use_position_encoding
        self.use_temporal_encoding = use_temporal_encoding

        # Embeddings per hop (adjacent weight tying)
        self.C = nn.ModuleList([nn.Embedding(self.V, self.d, padding_idx=pad_id) for _ in range(self.K)])
        self.A = nn.ModuleList([nn.Embedding(self.V, self.d, padding_idx=pad_id) for _ in range(self.K)])

        # A^{k+1} = C^k
        for k in range(1, self.K):
            self.A[k].weight = self.C[k - 1].weight

        # Question embedding B = A^1 (we reuse A[0])

        # Output projection (tied to final C embedding)
        self.W = nn.Linear(self.d, self.V, bias=False)
        if tie_out:
            self.W.weight = self.C[-1].weight

        # Temporal encoding (learned)
        if self.use_temporal_encoding:
            self.TA = nn.Embedding(self.I, self.d)
            self.TC = nn.Embedding(self.I, self.d)
        else:
            self.TA = None
            self.TC = None

        # Position encodings (fixed buffers)
        if self.use_position_encoding:
            self.register_buffer("pe_sentence", position_encoding_matrix(self.J, self.d, device=torch.device("cpu")))
            self.register_buffer("pe_question", position_encoding_matrix(self.Jq, self.d, device=torch.device("cpu")))
        else:
            self.pe_sentence = None
            self.pe_question = None

        self.reset_parameters()

    def reset_parameters(self) -> None:
        # Paper initializes with N(0, 0.1)
        for emb in list(self.A) + list(self.C):
            nn.init.normal_(emb.weight, mean=0.0, std=0.1)
            if emb.padding_idx is not None:
                with torch.no_grad():
                    emb.weight[emb.padding_idx].fill_(0.0)

        if self.use_temporal_encoding:
            nn.init.normal_(self.TA.weight, mean=0.0, std=0.1)
            nn.init.normal_(self.TC.weight, mean=0.0, std=0.1)

    def _encode(self, emb: nn.Embedding, tokens: torch.Tensor, pe: Optional[torch.Tensor]) -> torch.Tensor:
        if self.use_position_encoding:
            assert pe is not None
            pe = pe.to(tokens.device)
            return encode_pe(emb, tokens, pe)
        return encode_bow(emb, tokens)

    def forward(
        self,
        memories: torch.Tensor,
        questions: torch.Tensor,
        *,
        return_attn: bool = False,
    ):
        """
        memories: [B, I, J]
        questions: [B, Jq]
        """
        B = memories.size(0)

        u = self._encode(self.A[0], questions, self.pe_question)  # [B, d]

        attn_per_hop: List[torch.Tensor] = []
        pos_idx = torch.arange(self.I, device=memories.device)

        for k in range(self.K):
            m = self._encode(self.A[k], memories, self.pe_sentence)  # [B, I, d]
            c = self._encode(self.C[k], memories, self.pe_sentence)  # [B, I, d]

            if self.use_temporal_encoding:
                assert self.TA is not None and self.TC is not None
                ta = self.TA(pos_idx).unsqueeze(0)  # [1, I, d]
                tc = self.TC(pos_idx).unsqueeze(0)  # [1, I, d]
                m = m + ta
                c = c + tc

            scores = (m * u.unsqueeze(1)).sum(dim=-1)  # [B, I]
            p = F.softmax(scores, dim=-1)  # [B, I]

            o = torch.bmm(p.unsqueeze(1), c).squeeze(1)  # [B, d]
            u = u + o

            if return_attn:
                attn_per_hop.append(p.detach())

        logits = self.W(u)  # [B, V]
        if return_attn:
            return logits, attn_per_hop
        return logits


# Quick forward check
model = MemN2N(
    vocab_size=V,
    d=D_EMB,
    memory_size=mem_size,
    sentence_size=sent_size,
    question_size=q_size,
    hops=K_HOPS,
    pad_id=PAD_ID,
    use_position_encoding=USE_POSITION_ENCODING,
    use_temporal_encoding=USE_TEMPORAL_ENCODING,
).to(DEVICE)

m, q, y, tid = next(iter(train_loader))
logits, attn = model(m.to(DEVICE), q.to(DEVICE), return_attn=True)
print("logits:", logits.shape, "#hops attn:", len(attn), "attn[0]:", attn[0].shape)


logits: torch.Size([32, 183]) #hops attn: 3 attn[0]: torch.Size([32, 50])


## Training & evaluation

We train with cross-entropy loss on the answer word. We also apply gradient clipping (paper uses an $ell_2$ norm cap).


In [6]:
def _unpack_batch(batch):
    # batch is either (m, q, y) or (m, q, y, task_id)
    if len(batch) == 3:
        m, q, y = batch
        tid = None
    else:
        m, q, y, tid = batch
    return m, q, y, tid


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        m, q, y, _ = _unpack_batch(batch)
        m = m.to(DEVICE)
        q = q.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(m, q)
        loss = F.cross_entropy(logits, y)
        total_loss += loss.item() * y.size(0)
        preds = logits.argmax(dim=-1)
        total_correct += (preds == y).sum().item()
        total += y.size(0)
    return total_loss / max(1, total), total_correct / max(1, total)


@torch.no_grad()
def evaluate_per_task(model: nn.Module, loader: DataLoader, *, task_ids: List[int]) -> Dict[int, float]:
    """Return accuracy per task_id (only if task_id is provided by the dataset)."""
    model.eval()
    correct = {tid: 0 for tid in task_ids}
    total = {tid: 0 for tid in task_ids}

    for batch in loader:
        m, q, y, tid = _unpack_batch(batch)
        if tid is None:
            raise ValueError("Dataset does not provide task_id; cannot compute per-task metrics.")

        m = m.to(DEVICE)
        q = q.to(DEVICE)
        y = y.to(DEVICE)
        tid = tid.to(DEVICE)

        logits = model(m, q)
        preds = logits.argmax(dim=-1)

        for t in task_ids:
            mask = tid == t
            if mask.any():
                correct[t] += (preds[mask] == y[mask]).sum().item()
                total[t] += mask.sum().item()

    return {t: (correct[t] / total[t] if total[t] > 0 else float("nan")) for t in task_ids}


def train_model(
    *,
    d_emb: int,
    hops: int,
    use_pe: bool,
    use_temporal: bool,
    epochs: int = EPOCHS,
    lr: float = LR,
) -> MemN2N:
    model = MemN2N(
        vocab_size=V,
        d=d_emb,
        memory_size=mem_size,
        sentence_size=sent_size,
        question_size=q_size,
        hops=hops,
        pad_id=PAD_ID,
        use_position_encoding=use_pe,
        use_temporal_encoding=use_temporal,
    ).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_state = None
    best_valid_acc = -1.0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for batch in train_loader:
            m, q, y, _ = _unpack_batch(batch)
            m = m.to(DEVICE)
            q = q.to(DEVICE)
            y = y.to(DEVICE)

            opt.zero_grad(set_to_none=True)
            logits = model(m, q)
            loss = F.cross_entropy(logits, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            opt.step()

            running_loss += loss.item() * y.size(0)
            running_correct += (logits.argmax(dim=-1) == y).sum().item()
            running_total += y.size(0)

        tr_loss = running_loss / max(1, running_total)
        tr_acc = running_correct / max(1, running_total)
        va_loss, va_acc = evaluate(model, valid_loader)

        if va_acc > best_valid_acc:
            best_valid_acc = va_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"epoch {epoch:03d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | valid loss {va_loss:.4f} acc {va_acc:.3f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    te_loss, te_acc = evaluate(model, test_loader)
    print(f"test   | loss {te_loss:.4f} acc {te_acc:.3f}")

    per_task = evaluate_per_task(model, test_loader, task_ids=TASK_IDS)
    print("\nPer-task test accuracy:")
    for t in TASK_IDS:
        print(f"  task {t:02d}: {per_task[t]:.3f}")

    return model


In [7]:
def ids_to_tokens(ids: np.ndarray, itos: Dict[int, str], pad_id: int) -> List[str]:
    return [itos[int(i)] for i in ids if int(i) != pad_id]


def inspect_attention(
    model: MemN2N,
    *,
    sample_index: int = 0,
    topk: int = 3,
) -> None:
    model.eval()

    m = torch.from_numpy(Xte_m[sample_index : sample_index + 1]).to(DEVICE)
    q = torch.from_numpy(Xte_q[sample_index : sample_index + 1]).to(DEVICE)
    y = torch.from_numpy(yte[sample_index : sample_index + 1]).to(DEVICE)

    logits, attn = model(m, q, return_attn=True)
    pred = int(logits.argmax(dim=-1).item())
    gold = int(y.item())

    q_tokens = ids_to_tokens(Xte_q[sample_index], itos, PAD_ID)
    print("Q:", " ".join(q_tokens))
    print("gold:", itos[gold], "pred:", itos[pred])

    story_sentences = [ids_to_tokens(Xte_m[sample_index, i], itos, PAD_ID) for i in range(mem_size)]

    for hop, p in enumerate(attn, start=1):
        p_np = p[0].detach().cpu().numpy()
        best = np.argsort(-p_np)[:topk]
        print(f"\nHop {hop} (top {topk}):")
        for rank, idx in enumerate(best, start=1):
            sent = story_sentences[int(idx)]
            prob = float(p_np[int(idx)])
            print(f"  {rank}. i={int(idx):02d} p={prob:.3f} |", " ".join(sent))


# --- Training run (full pipeline on all 20 tasks) ---
cfg = {"hops": 3, "use_pe": True, "use_temporal": True}
print("\n===", cfg, "===")

model = train_model(
    d_emb=D_EMB,
    hops=cfg["hops"],
    use_pe=cfg["use_pe"],
    use_temporal=cfg["use_temporal"],
)

print(f"\n=== Attention inspection {cfg} ===")
inspect_attention(model, sample_index=0, topk=3)



=== {'hops': 3, 'use_pe': True, 'use_temporal': True} ===
epoch 001 | train loss 0.9798 acc 0.608 | valid loss 0.6057 acc 0.751
epoch 005 | train loss 0.3459 acc 0.851 | valid loss 0.3447 acc 0.848
epoch 010 | train loss 0.2882 acc 0.873 | valid loss 0.3076 acc 0.867
epoch 015 | train loss 0.2735 acc 0.882 | valid loss 0.3007 acc 0.872
epoch 020 | train loss 0.2679 acc 0.885 | valid loss 0.2996 acc 0.875
epoch 025 | train loss 0.2625 acc 0.888 | valid loss 0.2943 acc 0.876
epoch 030 | train loss 0.2598 acc 0.890 | valid loss 0.2947 acc 0.877
test   | loss 0.2900 acc 0.878

Per-task test accuracy:
  task 01: 1.000
  task 02: 0.981
  task 03: 0.860
  task 04: 1.000
  task 05: 0.974
  task 06: 0.938
  task 07: 0.895
  task 08: 0.932
  task 09: 0.958
  task 10: 0.902
  task 11: 1.000
  task 12: 1.000
  task 13: 1.000
  task 14: 1.000
  task 15: 1.000
  task 16: 0.456
  task 17: 0.612
  task 18: 0.907
  task 19: 0.141
  task 20: 1.000

=== Attention inspection {'hops': 3, 'use_pe': True, '

Results (numbers taken from the printed outputs above)

- After 30 epochs of joint training on `en-10k`, the run above reports **test loss 0.2900** and **test accuracy 0.878** (87.8%).
- Many tasks reach very high accuracy; the weakest ones in this printout are **task 16 (0.456)**, **task 19 (0.141)**, and **task 17 (0.612)**.

Attention (same printed inspection)

- For `where is john ?`, hop 1 puts **p = 0.995** on `john travelled to the hallway`.
- Hop 2 puts **p = 1.000** on the same sentence, and the prediction matches the gold answer `hallway`.
